# Notebook 3 — Model Training & Evaluation

This notebook builds, trains, and evaluates the deep learning model for IPL score prediction. It also includes an interactive widget for live predictions.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import tensorflow as tf
import keras
from sklearn.metrics import mean_absolute_error, mean_squared_error

%matplotlib inline
print('TensorFlow version:', tf.__version__)

## 1. Load Preprocessed Data

In [ ]:
X_train = np.load('../models/X_train_scaled.npy')
X_test  = np.load('../models/X_test_scaled.npy')
y_train = np.load('../models/y_train.npy')
y_test  = np.load('../models/y_test.npy')

label_encoders = joblib.load('../models/label_encoders.pkl')
scaler         = joblib.load('../models/scaler.pkl')

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')

## 2. Build the Neural Network

We use a simple 3-layer architecture. Huber loss is chosen over MSE because T20 innings can have high-scoring outliers that would otherwise dominate the gradient.

In [ ]:
model = keras.Sequential([
    keras.layers.Input(shape=(X_train.shape[1],)),
    keras.layers.Dense(512, activation='relu'),
    keras.layers.Dense(216, activation='relu'),
    keras.layers.Dense(1,   activation='linear')
])

huber_loss = tf.keras.losses.Huber(delta=1.0)
model.compile(optimizer='adam', loss=huber_loss)
model.summary()

## 3. Train the Model

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_test, y_test),
    verbose=1
)

## 4. Plot Training Loss

In [ ]:
loss_df = pd.DataFrame(history.history)

plt.figure(figsize=(9, 5))
plt.plot(loss_df['loss'],     label='Training Loss',   color='#0d2137', linewidth=2)
plt.plot(loss_df['val_loss'], label='Validation Loss', color='#ffa500', linewidth=2, linestyle='--')
plt.title('Training vs Validation Loss', fontsize=14, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Huber Loss')
plt.legend()
plt.tight_layout()
plt.savefig('../outputs/training_loss.png', dpi=150)
plt.show()

## 5. Evaluate the Model

In [ ]:
predictions = model.predict(X_test).flatten()

mae  = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print(f'Mean Absolute Error  (MAE) : {mae:.4f} runs')
print(f'Root Mean Squared Error    : {rmse:.4f} runs')

In [ ]:
# Actual vs Predicted scatter plot
plt.figure(figsize=(8, 6))
plt.scatter(y_test, predictions, alpha=0.3, color='steelblue', s=10)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', linewidth=1.5, label='Perfect prediction')
plt.xlabel('Actual Total')
plt.ylabel('Predicted Total')
plt.title('Actual vs Predicted Innings Total', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('../outputs/actual_vs_predicted.png', dpi=150)
plt.show()

## 6. Save the Model

In [ ]:
model.save('../models/ipl_model.h5')
print('Model saved to models/ipl_model.h5')

## 7. Interactive Prediction Widget

Use the dropdowns and input fields below to simulate a live match situation and get a predicted score.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

venue       = widgets.Dropdown(options=list(label_encoders['venue'].classes_),    description='Venue:')
batting_team = widgets.Dropdown(options=list(label_encoders['bat_team'].classes_), description='Batting Team:')
bowling_team = widgets.Dropdown(options=list(label_encoders['bowl_team'].classes_),description='Bowling Team:')
striker_name = widgets.Dropdown(options=list(label_encoders['batsman'].classes_),  description='Striker:')
bowler_name  = widgets.Dropdown(options=list(label_encoders['bowler'].classes_),   description='Bowler:')
runs_now     = widgets.IntText(value=78,  description='Runs So Far:')
wickets_now  = widgets.IntText(value=2,   description='Wickets:')
overs_now    = widgets.FloatText(value=10.0, description='Overs:')
striker_ind  = widgets.IntText(value=1,   description='Striker (0/1):')

for w in [venue, batting_team, bowling_team, striker_name, bowler_name,
          runs_now, wickets_now, overs_now, striker_ind]:
    w.style = {'description_width': 'initial'}

predict_btn = widgets.Button(description='Predict Score', button_style='primary')
output      = widgets.Output()

def predict_score(b):
    with output:
        clear_output()
        features = [
            label_encoders['bat_team'].transform([batting_team.value])[0],
            label_encoders['bowl_team'].transform([bowling_team.value])[0],
            label_encoders['venue'].transform([venue.value])[0],
            runs_now.value,
            wickets_now.value,
            overs_now.value,
            striker_ind.value,
            label_encoders['batsman'].transform([striker_name.value])[0],
            label_encoders['bowler'].transform([bowler_name.value])[0],
        ]
        X_input = np.array(features).reshape(1, -1)
        X_input = scaler.transform(X_input)
        pred    = model.predict(X_input, verbose=0)
        print(f'\n  ► Predicted Total Runs: {int(pred[0][0])}\n')

predict_btn.on_click(predict_score)

display(
    venue, batting_team, bowling_team, striker_name, bowler_name,
    runs_now, wickets_now, overs_now, striker_ind,
    predict_btn, output
)